# 🧱 LEGO Recognition: Unified Pipeline (Strategy C)
**Interactive Setup** – Elige un **Set** o una **Pieza** para entrenar el modelo de reconocimiento.
El sistema verificará automáticamente si ya existe un modelo en Google Drive para cada pieza.

---

In [ ]:
# =================================================================
# CELDA 0: Setup del Entorno (v3.16 Iron-Clad)
# =================================================================
import os, sys, shutil, json, zipfile, urllib.request, logging, subprocess
from pathlib import Path

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = os.path.exists('/kaggle')
PROJECT_ROOT = ''
DATASET_DIR = ''
LOG_FILE_PATH = ''
LDRAW_PATH = '/tmp/ldraw'
ADDON_PATH = '/tmp'
BLENDER_PATH = 'blender'

if IS_KAGGLE:
    print('Kaggle detectado | v3.16 Iron-Clad')
    WORKING_DIR = '/kaggle/working/proj'
    DATASET_DIR = '/kaggle/working/datasets'
    LOG_FILE_PATH = '/kaggle/working/pipeline_lego.log'
    zip_path = next(Path('/kaggle/input').rglob('*.zip'), None)
    if zip_path:
        os.makedirs(WORKING_DIR, exist_ok=True)
        with zipfile.ZipFile(zip_path, 'r') as z: z.extractall(WORKING_DIR)
        PROJECT_ROOT = WORKING_DIR
    else:
        src_dir = next(Path('/kaggle/input').rglob('src'), None)
        if src_dir: PROJECT_ROOT = str(src_dir.parent)
    if PROJECT_ROOT and os.path.isdir(PROJECT_ROOT):
        extracted = os.listdir(PROJECT_ROOT)
        if len(extracted) == 1 and os.path.isdir(os.path.join(PROJECT_ROOT, extracted[0])):
            PROJECT_ROOT = os.path.join(PROJECT_ROOT, extracted[0])
    os.system('pip install ultralytics pandas requests scikit-learn psutil ipywidgets pyyaml -q')
elif IS_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/Lego_Training'
    DATASET_DIR = '/content/datasets'
    LOG_FILE_PATH = '/content/pipeline_lego.log'

PROJECT_ROOT = os.path.abspath(PROJECT_ROOT or '')
if PROJECT_ROOT and PROJECT_ROOT not in sys.path: sys.path.insert(0, PROJECT_ROOT)
os.makedirs(DATASET_DIR, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
                    handlers=[logging.FileHandler(LOG_FILE_PATH, mode='w'), logging.StreamHandler()])
logger = logging.getLogger('LegoVision')

from src.utils.pipeline_timer import PipelineTimer
timer = PipelineTimer()
timer.detect_hardware()

LAUNCH_CONFIG = None
RENDER_ENGINE = 'CYCLES'
DRIVE_SERVICE = None
DRIVE_MODELS_FOLDER_ID = None

if PROJECT_ROOT:
    config_path = os.path.join(PROJECT_ROOT, 'config_train.json')
    if os.path.exists(config_path):
        with open(config_path, 'r') as f: LAUNCH_CONFIG = json.load(f)
        timer.log_config(LAUNCH_CONFIG)
        RENDER_ENGINE = LAUNCH_CONFIG.get('render_settings', {}).get('engine', 'CYCLES').upper()
    
    try:
        from src.utils.drive_manager import DriveManager
        # Use the primary account credentials
        token_path = os.path.join(PROJECT_ROOT, 'token_1973.pickle')
        creds_path = os.path.join(PROJECT_ROOT, 'credentials.json')
        if os.path.exists(token_path):
            dm = DriveManager(credentials_path=creds_path, token_path=token_path)
            if dm.authenticate():
                DRIVE_SERVICE = dm.service
                root_id = dm.ensure_folder('Lego_Training_75078')
                DRIVE_MODELS_FOLDER_ID = dm.ensure_folder('models', parent_id=root_id)
                logger.info(f'✅ Drive autenticado: {dm.account_email}')
    except Exception as e:
        logger.warning(f'⚠️ Drive no disponible para verificacion: {e}')

logger.info(f'Pipeline v3.16 | Motor: {RENDER_ENGINE} | Root: {PROJECT_ROOT}')


In [ ]:
# =====================================================================
# CELDA 1: Instalacion Blender + LDraw + Addon (v3.14 EGL)
# =====================================================================
def download_file(url, target_path):
    logger.info(f'Descargando {os.path.basename(target_path)}...')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, open(target_path, 'wb') as out:
        shutil.copyfileobj(response, out)

with timer.step('Install Blender + Dependencies'):
    if RENDER_ENGINE == 'EEVEE':
        BLENDER_PORTABLE_DIR = '/kaggle/working/blender-4.2.7-linux-x64'
        BLENDER_PATH = os.path.join(BLENDER_PORTABLE_DIR, 'blender')
        if not os.path.exists(BLENDER_PATH):
            logger.info('Descargando Blender 4.2 LTS Portable (EGL)...')
            blender_tarball = '/kaggle/working/blender.tar.xz'
            download_file(
                'https://mirror.clarkson.edu/blender/release/Blender4.2/blender-4.2.7-linux-x64.tar.xz',
                blender_tarball
            )
            logger.info('Extrayendo Blender...')
            subprocess.run(['tar', '-xf', blender_tarball, '-C', '/kaggle/working/'], check=True)
            os.remove(blender_tarball)
        os.environ['__EGL_VENDOR_LIBRARY_FILENAMES'] = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
        os.environ['DISPLAY'] = ''
        logger.info(f'EEVEE Mode: Blender portable={BLENDER_PATH} | EGL OK')
    else:
        logger.info('CYCLES Mode: Instalando Blender via apt...')
        os.system('apt update -y > /dev/null 2>&1')
        os.system('apt install -y blender libglu1-mesa-dev libxi-dev libxmu-dev libgl1-mesa-glx > /dev/null 2>&1')
        BLENDER_PATH = 'blender'
        logger.info('CYCLES Mode: Blender system listo')

    if not os.path.exists('/tmp/ldraw'):
        ldraw_zip = '/tmp/ldraw_complete.zip'
        download_file('https://library.ldraw.org/library/updates/complete.zip', ldraw_zip)
        with zipfile.ZipFile(ldraw_zip, 'r') as z: z.extractall('/tmp/ldraw_temp')
        temp_list = os.listdir('/tmp/ldraw_temp')
        if len(temp_list) == 1 and temp_list[0].lower() == 'ldraw':
            shutil.move('/tmp/ldraw_temp/ldraw', '/tmp/ldraw')
            shutil.rmtree('/tmp/ldraw_temp')
        else:
            shutil.move('/tmp/ldraw_temp', '/tmp/ldraw')

    if not os.path.exists('/tmp/ImportLDraw'):
        addon_zip = '/tmp/importldraw.zip'
        download_file('https://github.com/TobyLobster/ImportLDraw/releases/download/v1.2.2/importldraw1.2.2.zip', addon_zip)
        with zipfile.ZipFile(addon_zip, 'r') as z: z.extractall('/tmp')
        inner = [d for d in os.listdir('/tmp') if 'importldraw' in d.lower() and os.path.isdir(os.path.join('/tmp', d))][0]
        shutil.move(os.path.join('/tmp', inner), '/tmp/ImportLDraw')

ADDON_PATH = '/tmp'
LDRAW_PATH = '/tmp/ldraw'
logger.info(f'Listo. Blender={BLENDER_PATH} | LDraw={LDRAW_PATH}')


In [ ]:
# =====================================================================
# CELDA 2: 🎯 Resolución de Piezas (Auto-Config Bypasser)
# =====================================================================
import ipywidgets as widgets
from IPython.display import display, clear_output
from src.logic.part_resolver import resolve_set, resolve_piece

RESOLVED_PARTS = []
RENDER_ENGINE = 'EEVEE'
NUM_IMAGES = 150
SET_ID = "custom"

if LAUNCH_CONFIG:
    print("🚀 Usando configuración automática del Launchpad...")
    target_parts = LAUNCH_CONFIG.get('target_parts', [])
    RENDER_ENGINE = LAUNCH_CONFIG.get('render_settings', {}).get('engine', 'EEVEE')
    NUM_IMAGES = LAUNCH_CONFIG.get('render_settings', {}).get('num_images', 150)
    SET_ID = LAUNCH_CONFIG.get('session_reference', 'custom').split('_')[0]
    
    for p_id in target_parts:
        RESOLVED_PARTS.extend(resolve_piece(p_id))
    
    print(f"   • Motor: {RENDER_ENGINE}")
    print(f"   • Imágenes: {NUM_IMAGES}")
    print(f"   • Piezas a procesar: {[p['ldraw_id'] for p in RESOLVED_PARTS]}")
else:
    # Mantenemos el UI por si alguien quiere usar el notebook manualmente
    print("⌨️ No se detectó config_train.json. Iniciando modo interactivo...")
    # ... (rest of the ipywidgets code if needed, but the user wants to eliminate it/fix it)
    # For simplicity and following user request to 'eliminate this part', we'll just put a minimal warning or keep it but only if no config.
    print("⚠️ Modo interactivo desactivado en esta versión. Por favor usa el Launchpad local.")


In [ ]:
# =====================================================================
# CELDA 3: ✅ Verificación y Filtrado
# =====================================================================
from src.logic.model_registry import get_training_status, filter_pending

if not RESOLVED_PARTS:
    raise Exception("❌ No hay piezas para procesar.")

status_list = get_training_status(RESOLVED_PARTS, PROJECT_ROOT, drive_service=DRIVE_SERVICE, drive_models_folder_id=DRIVE_MODELS_FOLDER_ID)
PARTS_TO_TRAIN = filter_pending(status_list)
TARGET_PART_IDS = [p['ldraw_id'] for p in PARTS_TO_TRAIN]

print(f"\n─── RESUMEN DE MODELOS ───")
for s in status_list:
    status = "✅" if s['is_complete'] else "❌"
    print(f"{status} {s['ldraw_id']}: {s['name']}")


In [ ]:
# =================================================================
# CELDA 4: Generacion de Dataset (Multi-GPU High Detail)
# =================================================================
import subprocess, concurrent.futures, torch, time, psutil
import shutil

ADDON_PATH = '/tmp'
LDRAW_PATH = '/tmp/ldraw'
logger.info(f'Listo. Blender={BLENDER_PATH} | LDraw={LDRAW_PATH}')

# --- Blender Kernel Restoration ---
def restore_kernels():
    kernel_found = False
    potential_dirs = ['/kaggle/input/blender-kernels', PROJECT_ROOT]
    for kernel_dir in potential_dirs:
        if not os.path.exists(kernel_dir): continue
        cycles_bin = os.path.join(kernel_dir, 'cycles_kernels.bin')
        if os.path.exists(cycles_bin):
            dest = os.path.expanduser('~/.cache/cycles')
            os.makedirs(dest, exist_ok=True)
            try:
                shutil.unpack_archive(cycles_bin, dest, 'zip')
                print(f'\n🚀 [KERNEL] Cycles Kernels inyectados con éxito desde {kernel_dir}')
                kernel_found = True
            except Exception as e:
                print(f'❌ Error descomprimiendo Cycles Kernels: {e}')
        optix_bin = os.path.join(kernel_dir, 'optix_cache.bin')
        if os.path.exists(optix_bin):
            dest = f"/var/tmp/OptixCache_{os.environ.get('USER', 'root')}"
            os.makedirs(dest, exist_ok=True)
            try:
                shutil.unpack_archive(optix_bin, dest, 'zip')
                print(f'🚀 [KERNEL] OptiX Cache inyectado con éxito desde {kernel_dir}')
                kernel_found = True
            except Exception as e:
                print(f'❌ Error descomprimiendo OptiX Cache: {e}')
    if not kernel_found:
        print(f'\n⚠️ [ATENCIÓN] No se encontraron kernels precompilados en {potential_dirs}. Blender perderá minutos compilándolos.')
restore_kernels()

total_images_to_render = LAUNCH_CONFIG.get('render_settings', {}).get('num_images', 200) if LAUNCH_CONFIG else 200
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
num_workers = 3 # Hardcoded to 3 as requested
images_per_worker = max(1, total_images_to_render // num_workers)
logger.info(f'GPUs: {NUM_GPUS} | Workers: {num_workers} | Imgs/worker: {images_per_worker}')

def run_render_worker(worker_id):
    gpu_id = worker_id % NUM_GPUS
    worker_cfg = {
        'set_id': SET_ID, 'parts': RESOLVED_PARTS, 'num_images': images_per_worker,
        'offset_idx': worker_id * images_per_worker, 'output_base': os.path.join(DATASET_DIR, SET_ID),
        'assets_dir': os.path.join(PROJECT_ROOT, 'assets'), 'ldraw_path': LDRAW_PATH,
        'addon_path': ADDON_PATH, 'render_engine': RENDER_ENGINE
    }
    cfg_path = f'/tmp/render_cfg_{worker_id}.json'
    with open(cfg_path, 'w') as f: json.dump(worker_cfg, f)
    env = os.environ.copy()
    env['CUDA_VISIBLE_DEVICES'] = str(gpu_id)
    env['OPTIX_VISIBLE_DEVICES'] = str(gpu_id)
    env['PYTHONPATH'] = PROJECT_ROOT
    log_file = f'/tmp/worker_{worker_id}.log'
    with open(log_file, 'w') as f_out:
        subprocess.run([BLENDER_PATH, '--background', '--python',
            os.path.join(PROJECT_ROOT, 'src', 'blender_scripts', 'scene_setup.py'),
            '--', cfg_path], stdout=f_out, stderr=subprocess.STDOUT, env=env)

images_dir = os.path.join(DATASET_DIR, SET_ID, 'images')
os.makedirs(images_dir, exist_ok=True)

with timer.step('Blender Render Workers'):
    print('\n🚀 INICIANDO RENDER | ' + str(num_workers) + ' Workers | ' + RENDER_ENGINE)
    print('=' * 100)
    start_t = time.time()
    with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(run_render_worker, i) for i in range(num_workers)]
        while any(f.running() for f in futures):
            curr_imgs = len(os.listdir(images_dir)) if os.path.exists(images_dir) else 0
            elapsed = time.time() - start_t
            worker_status = []
            for i in range(num_workers):
                tail = 'Iniciando...' 
                log_f = f'/tmp/worker_{i}.log'
                if os.path.exists(log_f):
                    try:
                        with open(log_f, 'r') as f:
                            lines = f.readlines()
                            if lines: tail = lines[-1].strip()[-40:]
                    except: pass
                worker_status.append(f'W{i}: {tail}')
            cpu, ram = psutil.cpu_percent(), psutil.virtual_memory().percent
            vram = torch.cuda.memory_reserved(0)/1e9 if torch.cuda.is_available() else 0
            fps = curr_imgs/elapsed if elapsed > 0 else 0
            pct = (curr_imgs/total_images_to_render)*100
            print(f'[{pct:>3.0f}%] {curr_imgs}/{total_images_to_render} | {fps:.1f} FPS | CPU {cpu}% | RAM {ram}% | VRAM {vram:.1f}GB | {int(elapsed)}s')
            for s in worker_status: print(f'  ↳ {s}')
            print('\033[F' * (num_workers + 2), end='', flush=True)
            time.sleep(3)
    print('\n' * (num_workers + 1) + '=' * 100)
    print(f'✅ Render completado: {len(os.listdir(images_dir))} imágenes.')


In [ ]:
# =================================================================
# CELDA 5: YOLO Training (Stage 1 Localization)
# =================================================================
from ultralytics import YOLO
import torch, time as _time

if not PARTS_TO_TRAIN:
    print('No hay piezas pendientes. Saltando entrenamiento.')
else:
    dataset_path = os.path.join(DATASET_DIR, SET_ID)
    os.makedirs(dataset_path, exist_ok=True)
    data_yaml = os.path.join(dataset_path, 'data.yaml')

    results_dir = '/kaggle/working/results'
    base_model_path = 'yolo11n-obb.pt'  # Default starting point
    models_dir = os.path.join(PROJECT_ROOT, 'models', 'yolo_model')
    if os.path.exists(models_dir):
        pt_files = [f for f in os.listdir(models_dir) if f.endswith('.pt')]
        if pt_files:
            if 'detector_universal.pt' in pt_files:
                base_model_path = os.path.join(models_dir, 'detector_universal.pt')
            else:
                base_model_path = os.path.join(models_dir, sorted(pt_files)[-1])
            logger.info(f'🧠 Entrenando YOLO sobre pesos existentes: {os.path.basename(base_model_path)}')
        else:
            logger.info('🧠 No hay pesos previos, empezando desde yolo11n-obb.pt')
    model = YOLO(base_model_path, task='obb')

    num_gpus = torch.cuda.device_count()
    train_device = list(range(num_gpus)) if num_gpus > 1 else (0 if num_gpus == 1 else 'cpu')
    train_batch = (8 * num_gpus) if num_gpus > 0 else 8

    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    torch.cuda.empty_cache()

    _t_epoch = [_time.time()]
    def _on_epoch_end(trainer):
        elapsed = _time.time() - _t_epoch[0]
        m = trainer.metrics or {}
        vram = torch.cuda.memory_reserved(0)/1e9 if torch.cuda.is_available() else 0
        print(f'\n[EPOCH {trainer.epoch+1}/{trainer.epochs}] Time: {elapsed:.1f}s | Box: {m.get("train/box_loss",0):.4f} | Cls: {m.get("train/cls_loss",0):.4f} | mAP50: {m.get("metrics/mAP50(B)",0):.4f} | VRAM: {vram:.1f}GB', flush=True)
    model.add_callback('on_fit_epoch_end', _on_epoch_end)

    # MODO DDP GPU NATIVO: PyTorch Local Rank spawnea la pipeline eficientemente
    # Configuraciones Agresivas (Multi-GPU)
    with timer.step('YOLO T4 Dual-GPU Training'):
        print(f'🚀 Iniciando entrenamiento Multi-GPU en dispositivos: {train_device} con batch={train_batch}')
        model.train(data=data_yaml, epochs=25, patience=5, imgsz=960, project=results_dir, name=f'yolo11_{SET_ID}',
                    batch=train_batch, device=train_device, workers=8, cache=True,
                    amp=True, optimizer='auto', verbose=False)


In [ ]:
# =================================================================
# CELDA 6: Vector Index Generation (Stage 2 Identification)
# =================================================================
try:
    from src.logic.build_reference_index import build_index
    dataset_path = os.path.join(DATASET_DIR, SET_ID)
    # Now output is a folder to hold individual .pkl files
    indices_dir = '/kaggle/working/indices'
    os.makedirs(indices_dir, exist_ok=True)
    
    with timer.step('Vector Index Generation'):
        print(f'🚀 Generando Indices individuales por pieza para {SET_ID}...')
        build_index(dataset_path, indices_dir)
        print(f'✅ Indices generados en: {indices_dir}')
except Exception as e:
    logger.error(f'❌ Error generando Indices de Vectores: {e}')


In [ ]:
# =================================================================
# CELDA 7: Extracción y Sincronización a Drive (Sin Zips)
# =================================================================
import os, json
PROJECT_ROOT = '/kaggle/working/lego_recognition_system'
LAUNCH_CONFIG = None
SET_ID = 'unknown'
config_path = os.path.join(PROJECT_ROOT, 'config_train.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f: LAUNCH_CONFIG = json.load(f)
    SET_ID = LAUNCH_CONFIG.get('session_reference', 'unknown')
session_ref = SET_ID
yolo_out = f'/kaggle/working/results/yolo11_{SET_ID}'
report_path = '/kaggle/working/performance_report.json'

timer.save_report(report_path)

with timer.step('Drive Sync & Output (Iron-Clad)'):
    if DRIVE_SERVICE and DRIVE_MODELS_FOLDER_ID:
        try:
            from src.utils.drive_manager import DriveManager
            dm = DriveManager(credentials_path=os.path.join(PROJECT_ROOT, 'credentials.json'), 
                             token_path=os.path.join(PROJECT_ROOT, 'token_1973.pickle'))
            if dm.authenticate():
                logger.info('☁️ Conectando a Google Drive...')
                
                # 1. Ensure subfolders exist inside models folder
                folder_yolo = dm.ensure_folder('yolo_model', parent_id=DRIVE_MODELS_FOLDER_ID)
                folder_piezas = dm.ensure_folder('piezas_vectores', parent_id=DRIVE_MODELS_FOLDER_ID)
                
                # 2. Subir YOLO (.pt)
                weights_path = os.path.join(yolo_out, 'weights', 'best.pt')
                if os.path.exists(weights_path):
                    import datetime
                    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
                    dm.upload_file(weights_path, folder_yolo, remote_name=f'detector_{ts}.pt', overwrite=True)
                    logger.info(f'✅ Detector subido a yolo_model/detector_{ts}.pt')
                
                # 3. Subir Indices (.pkl individuales)
                indices_src = '/kaggle/working/indices'
                if os.path.exists(indices_src):
                    pk_count = 0
                    for f in os.listdir(indices_src):
                        if f.endswith('.pkl'):
                            dm.upload_file(os.path.join(indices_src, f), folder_piezas, remote_name=f, overwrite=True)
                            pk_count += 1
                    logger.info(f'✅ {pk_count} archivos .pkl subidos a piezas_vectores/')
                
                # 4. Logs
                if os.path.exists(report_path):
                    dm.upload_file(report_path, DRIVE_MODELS_FOLDER_ID, overwrite=True)
                if os.path.exists(LOG_FILE_PATH):
                    dm.upload_file(LOG_FILE_PATH, DRIVE_MODELS_FOLDER_ID, overwrite=True)
                
                logger.info('🏁 Sincronización a Drive Completada con éxito.')
        except Exception as e:
            logger.error(f'⚠️ Error subiendo a Drive: {e}')
    else:
        logger.warning('⚠️ DRIVE_SERVICE no configurado. No se subirán modelos.')
